# TCGA-BRCA Expression Data Integration

**Date:** February 22, 2026  
**Session:** 1.2 - TCGA Data Organization  
**Objective:** Extract TPM values from existing STAR counts files and integrate with clinical data

## Data Source
- **Location:** `D:\Projects\brca-precision\data\raw\gdc\brca_rnaseq\rnaseq_star_counts\`
- **Files:** 1,096 STAR gene counts files (one per sample)
- **Format:** TSV with TPM values in column `tpm_unstranded`
- **Genes:** ~60,000 genes (GENCODE v36)

## What We're Extracting
- Gene expression matrix: genes × samples (TPM values)
- Linking with existing clinical data (105 variables)
- Linking with PAM50 labels (1,095 cases)

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm import tqdm
import glob

# Paths
PROJECT_ROOT = Path.cwd().parent
NEW_PROJECT_ROOT = Path(r"D:\Projects\tcga-metabric-treatment-ai")

# Old TCGA data location
OLD_TCGA_DIR = Path(r"D:\Projects\brca-precision")
RNASEQ_DIR = OLD_TCGA_DIR / "data" / "raw" / "gdc" / "brca_rnaseq" / "rnaseq_star_counts"

# New project data directory
TCGA_DATA_DIR = NEW_PROJECT_ROOT / "data" / "raw" / "tcga"
TCGA_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ RNA-seq source: {RNASEQ_DIR}")
print(f"✓ Target directory: {TCGA_DATA_DIR}")
print(f"✓ Project root: {NEW_PROJECT_ROOT}")

✓ RNA-seq source: D:\Projects\brca-precision\data\raw\gdc\brca_rnaseq\rnaseq_star_counts
✓ Target directory: D:\Projects\tcga-metabric-treatment-ai\data\raw\tcga
✓ Project root: D:\Projects\tcga-metabric-treatment-ai


In [2]:
print("Finding STAR count files...")
print("=" * 80)

# Find all .tsv files in subdirectories
star_files = list(RNASEQ_DIR.glob("*/*.tsv"))

print(f"Found {len(star_files):,} STAR count files")
print(f"\nExample file: {star_files[0].name}")
print(f"Directory structure: {star_files[0].parent.name}")

Finding STAR count files...
Found 1,095 STAR count files

Example file: ba295155-272e-43eb-9d6a-e4c9c392e68b.rna_seq.augmented_star_gene_counts.tsv
Directory structure: 0019c951-16c5-48d0-85c8-58d96b12d330


In [3]:
print("\nExtracting TPM values from all samples...")
print("=" * 80)
print("This will take 3-5 minutes - processing 1,096 files...")

# Storage for expression data
expression_data = {}
gene_info = None

# Process each file
for file_path in tqdm(star_files, desc="Processing samples"):
    # Read file
    df = pd.read_csv(file_path, sep='\t', comment='#')
    
    # Get sample ID from directory name (GDC file UUID)
    sample_id = file_path.parent.name
    
    # Extract TPM values (skip the N_unmapped, N_multimapping, etc. rows)
    df_genes = df[df['gene_id'].str.startswith('ENSG')].copy()
    
    # Store gene info (only need to do this once)
    if gene_info is None:
        gene_info = df_genes[['gene_id', 'gene_name', 'gene_type']].copy()
    
    # Extract TPM values
    expression_data[sample_id] = df_genes['tpm_unstranded'].values

print(f"\n✓ Processed {len(expression_data):,} samples")
print(f"✓ Genes per sample: {len(gene_info):,}")


Extracting TPM values from all samples...
This will take 3-5 minutes - processing 1,096 files...


Processing samples: 100%|██████████| 1095/1095 [01:46<00:00, 10.28it/s]


✓ Processed 1,095 samples
✓ Genes per sample: 60,660


In [4]:
print("\nBuilding expression matrix...")
print("=" * 80)

# Create DataFrame
df_expression = pd.DataFrame(expression_data)
df_expression.index = gene_info['gene_name'].values

print(f"Expression matrix shape: {df_expression.shape}")
print(f"  Genes: {df_expression.shape[0]:,}")
print(f"  Samples: {df_expression.shape[1]:,}")

# Add gene metadata
df_expression.insert(0, 'gene_id', gene_info['gene_id'].values)
df_expression.insert(1, 'gene_type', gene_info['gene_type'].values)

print(f"\nFinal shape with metadata: {df_expression.shape}")
print(f"\nFirst few rows and columns:")
df_expression.iloc[:5, :8]


Building expression matrix...
Expression matrix shape: (60660, 1095)
  Genes: 60,660
  Samples: 1,095

Final shape with metadata: (60660, 1097)

First few rows and columns:


,gene_id,gene_type,0019c951-16c5-48d0-85c8-58d96b12d330,0022cd20-f64f-4773-b9ff-a3de0b71b259,00469928-b243-4cae-acd7-134508e99ceb,0081f507-b104-4214-9ea1-31dd69130991,0094f9d0-45ec-4aad-bca0-71c60bdd7113,00b13ccf-ad7c-4613-8366-7c583a399691
TSPAN6,ENSG00000000003.15,protein_coding,56.2216,28.5350,54.5178,90.7437,30.4434,26.7247
TNMD,ENSG00000000005.6,protein_coding,0.2768,5.1690,0.3343,0.6843,0.1295,5.0735
DPM1,ENSG00000000419.13,protein_coding,126.9161,101.9253,141.3968,89.1934,159.3406,96.1337
SCYL3,ENSG00000000457.14,protein_coding,25.4778,11.2845,8.5928,14.2512,19.2226,12.7431
C1orf112,ENSG00000000460.17,protein_coding,15.4251,3.6297,8.7852,3.6424,5.3065,5.2037


In [5]:
print("\nLinking with existing TCGA clinical data...")
print("=" * 80)

# Load existing clinical data
CLINICAL_FILE = OLD_TCGA_DIR / "analyses" / "gdc-risk-inventory" / "results" / "brca_gdc_clinical_inventory_completed.csv"

if CLINICAL_FILE.exists():
    df_tcga_clinical = pd.read_csv(CLINICAL_FILE)
    print(f"✓ Loaded clinical data: {df_tcga_clinical.shape}")
    print(f"  Variables: {df_tcga_clinical.shape[1]}")
else:
    print(f"❌ Clinical file not found: {CLINICAL_FILE}")

# Load PAM50 labels
PAM50_FILE = OLD_TCGA_DIR / "analyses" / "pam50-subtyping" / "results" / "brca_subtyping" / "tables" / "brca_wsi_pam50_case_labels.csv"

if PAM50_FILE.exists():
    df_pam50 = pd.read_csv(PAM50_FILE)
    print(f"✓ Loaded PAM50 labels: {df_pam50.shape}")
else:
    print(f"❌ PAM50 file not found: {PAM50_FILE}")

# Show sample IDs format
print(f"\nSample ID formats:")
print(f"  Expression samples (GDC UUID): {list(df_expression.columns[2:5])}")
print(f"  Clinical case_id: {df_tcga_clinical['case_id'].iloc[:3].tolist()}")
print(f"  PAM50 case_id: {df_pam50['case_id'].iloc[:3].tolist()}")


Linking with existing TCGA clinical data...
✓ Loaded clinical data: (1095, 105)
  Variables: 105
✓ Loaded PAM50 labels: (1095, 3)

Sample ID formats:
  Expression samples (GDC UUID): ['0019c951-16c5-48d0-85c8-58d96b12d330', '0022cd20-f64f-4773-b9ff-a3de0b71b259', '00469928-b243-4cae-acd7-134508e99ceb']
  Clinical case_id: ['001cef41-ff86-4d3f-a140-a647ac4b10a1', '0045349c-69d9-4306-a403-c9c1fa836644', '00807dae-9f4a-4fd1-aac2-82eb11bf2afb']
  PAM50 case_id: ['001cef41-ff86-4d3f-a140-a647ac4b10a1', '0045349c-69d9-4306-a403-c9c1fa836644', '00807dae-9f4a-4fd1-aac2-82eb11bf2afb']


In [6]:
print("Looking for GDC manifest file...")
print("=" * 80)

# Check for manifest in download directory
manifest_files = list(RNASEQ_DIR.parent.glob("*manifest*.txt")) + \
                 list(RNASEQ_DIR.parent.glob("*.tsv")) + \
                 list(RNASEQ_DIR.glob("*manifest*"))

if manifest_files:
    print(f"Found {len(manifest_files)} potential manifest files:")
    for f in manifest_files:
        print(f"  - {f}")
else:
    print("No manifest found - will query GDC API for file→case mapping")
    print("\nQuerying GDC API...")
    
    import requests
    
    # Get file UUIDs from expression data
    file_uuids = list(df_expression.columns[2:])[:5]  # Test with first 5
    
    # Query GDC for each file
    base_url = "https://api.gdc.cancer.gov/files"
    
    for file_id in file_uuids:
        params = {
            'filters': f'{{"op":"=","content":{{"field":"file_id","value":"{file_id}"}}}}',
            'fields': 'file_id,cases.case_id',
            'size': 1
        }
        response = requests.get(base_url, params=params)
        data = response.json()
        
        if data['data']['hits']:
            hit = data['data']['hits'][0]
            case_id = hit['cases'][0]['case_id'] if hit.get('cases') else 'NO_CASE'
            print(f"{file_id} → {case_id}")

Looking for GDC manifest file...
No manifest found - will query GDC API for file→case mapping

Querying GDC API...
0019c951-16c5-48d0-85c8-58d96b12d330 → 6a186809-3422-41d0-83d2-867145830936
0022cd20-f64f-4773-b9ff-a3de0b71b259 → c2a742fe-3e8b-4210-85a6-7191a1123609
00469928-b243-4cae-acd7-134508e99ceb → 5b2a4f11-ca46-4974-9420-59b4820920bf
0081f507-b104-4214-9ea1-31dd69130991 → 23b7aaea-1119-4b10-aa1a-0ae255d2f2a6
0094f9d0-45ec-4aad-bca0-71c60bdd7113 → 4922cddc-575c-4b8a-8245-ce5f6876760c


In [7]:
print("\nMapping all 1,095 files to case IDs...")
print("=" * 80)
print("This will take 2-3 minutes (querying GDC API)...")

import time

# Get all file UUIDs
file_uuids = list(df_expression.columns[2:])  # Skip gene_id and gene_type

# Create mapping
file_to_case = {}
failed = []

for i, file_id in enumerate(tqdm(file_uuids, desc="Querying GDC")):
    try:
        params = {
            'filters': f'{{"op":"=","content":{{"field":"file_id","value":"{file_id}"}}}}',
            'fields': 'file_id,cases.case_id',
            'size': 1
        }
        response = requests.get(base_url, params=params)
        data = response.json()
        
        if data['data']['hits'] and data['data']['hits'][0].get('cases'):
            case_id = data['data']['hits'][0]['cases'][0]['case_id']
            file_to_case[file_id] = case_id
        else:
            failed.append(file_id)
            
        # Rate limiting - be nice to GDC API
        if (i + 1) % 100 == 0:
            time.sleep(1)
            
    except Exception as e:
        print(f"\nError on {file_id}: {e}")
        failed.append(file_id)

print(f"\n✓ Mapped: {len(file_to_case):,} files")
print(f"✗ Failed: {len(failed)} files")

if len(file_to_case) > 0:
    print(f"\nSample mapping:")
    for file_id, case_id in list(file_to_case.items())[:3]:
        print(f"  {file_id} → {case_id}")


Mapping all 1,095 files to case IDs...
This will take 2-3 minutes (querying GDC API)...


Querying GDC: 100%|██████████| 1095/1095 [14:07<00:00,  1.29it/s]


✓ Mapped: 1,095 files
✗ Failed: 0 files

Sample mapping:
  0019c951-16c5-48d0-85c8-58d96b12d330 → 6a186809-3422-41d0-83d2-867145830936
  0022cd20-f64f-4773-b9ff-a3de0b71b259 → c2a742fe-3e8b-4210-85a6-7191a1123609
  00469928-b243-4cae-acd7-134508e99ceb → 5b2a4f11-ca46-4974-9420-59b4820920bf


In [8]:
print("\nRenaming expression columns from file IDs to case IDs...")
print("=" * 80)

# Create new column names mapping
column_rename = {file_id: case_id for file_id, case_id in file_to_case.items()}

# Rename columns (keep gene_id and gene_type as-is)
df_expression_renamed = df_expression.rename(columns=column_rename)

print(f"✓ Expression matrix: {df_expression_renamed.shape}")
print(f"  Genes: {df_expression_renamed.shape[0]:,}")
print(f"  Samples (now as case IDs): {df_expression_renamed.shape[1] - 2:,}")

print(f"\nColumn names now:")
print(f"  First 5 samples: {list(df_expression_renamed.columns[2:7])}")


Renaming expression columns from file IDs to case IDs...
✓ Expression matrix: (60660, 1097)
  Genes: 60,660
  Samples (now as case IDs): 1,095

Column names now:
  First 5 samples: ['6a186809-3422-41d0-83d2-867145830936', 'c2a742fe-3e8b-4210-85a6-7191a1123609', '5b2a4f11-ca46-4974-9420-59b4820920bf', '23b7aaea-1119-4b10-aa1a-0ae255d2f2a6', '4922cddc-575c-4b8a-8245-ce5f6876760c']


In [9]:
print("\nMerging TCGA datasets...")
print("=" * 80)

# Merge clinical and PAM50
df_tcga_merged = df_tcga_clinical.merge(
    df_pam50,
    on='case_id',
    how='inner'
)

print(f"✓ Clinical + PAM50: {df_tcga_merged.shape}")

# Check overlap with expression data
expression_cases = set(df_expression_renamed.columns[2:])
clinical_cases = set(df_tcga_merged['case_id'])

overlap = expression_cases & clinical_cases
print(f"\nData overlap:")
print(f"  Expression cases: {len(expression_cases):,}")
print(f"  Clinical cases: {len(clinical_cases):,}")
print(f"  Overlap: {len(overlap):,}")

if len(overlap) != len(expression_cases):
    print(f"  ⚠️ Missing from clinical: {len(expression_cases - clinical_cases)}")
if len(overlap) != len(clinical_cases):
    print(f"  ⚠️ Missing from expression: {len(clinical_cases - expression_cases)}")


Merging TCGA datasets...
✓ Clinical + PAM50: (1095, 107)

Data overlap:
  Expression cases: 1,095
  Clinical cases: 1,095
  Overlap: 1,095


In [10]:
print("\nSaving TCGA data...")
print("=" * 80)

PROCESSED_DIR = NEW_PROJECT_ROOT / 'data' / 'processed'

# Save clinical + PAM50
clinical_file = PROCESSED_DIR / 'tcga_clinical.csv'
df_tcga_merged.to_csv(clinical_file, index=False)
print(f"✓ Saved: {clinical_file}")
print(f"  Shape: {df_tcga_merged.shape}")

# Save expression (compressed)
expression_file = PROCESSED_DIR / 'tcga_expression.tsv.gz'
df_expression_renamed.to_csv(expression_file, sep='\t', index=False, compression='gzip')
print(f"✓ Saved: {expression_file}")
print(f"  Shape: {df_expression_renamed.shape}")
print(f"  Size: {expression_file.stat().st_size / 1e6:.1f} MB")

# Save file→case mapping for future reference
mapping_file = PROCESSED_DIR / 'tcga_file_to_case_mapping.csv'
pd.DataFrame(list(file_to_case.items()), columns=['file_id', 'case_id']).to_csv(mapping_file, index=False)
print(f"✓ Saved: {mapping_file}")

print("\n" + "="*80)
print("SESSION 1.2 COMPLETE")
print("="*80)


Saving TCGA data...
✓ Saved: D:\Projects\tcga-metabric-treatment-ai\data\processed\tcga_clinical.csv
  Shape: (1095, 107)
✓ Saved: D:\Projects\tcga-metabric-treatment-ai\data\processed\tcga_expression.tsv.gz
  Shape: (60660, 1097)
  Size: 111.9 MB
✓ Saved: D:\Projects\tcga-metabric-treatment-ai\data\processed\tcga_file_to_case_mapping.csv

SESSION 1.2 COMPLETE


## ✅ Session 1.2 Complete: TCGA-BRCA Expression Integration

**Date:** February 22, 2026  
**Duration:** ~1.5 hours (including GDC API queries)

### Data Integrated

**TCGA Expression Data:**
- 60,660 genes × 1,095 samples
- Format: TPM (Transcripts Per Million)
- Source: Existing STAR counts from brca-precision project
- Platform: RNA-seq (Illumina)

**TCGA Clinical + PAM50:**
- 1,095 patients × 107 variables
- Sources combined:
  - Clinical data (105 variables from GDC inventory)
  - PAM50 subtypes (from previous classification)
- **Perfect overlap:** 100% of expression samples have clinical data

### Technical Achievement
- ✅ Reused existing RNA-seq data (saved 2-3 hours of downloading)
- ✅ Queried GDC API for 1,095 file→case mappings
- ✅ Merged expression + clinical + PAM50 seamlessly
- ✅ No data loss in merging

### Files Created
- `data/processed/tcga_clinical.csv` (1,095 × 107)
- `data/processed/tcga_expression.tsv.gz` (60,660 × 1,097, 111.9 MB)
- `data/processed/tcga_file_to_case_mapping.csv` (reference)

### Data Comparison: TCGA vs METABRIC

| Feature | TCGA | METABRIC |
|---------|------|----------|
| Patients | 1,095 | 2,509 |
| Genes | 60,660 | 20,603 |
| Platform | RNA-seq | Microarray |
| Clinical vars | 107 | 36 |
| PAM50 coverage | 100% | 99.8% |
| RFS data | 18.6% | 95-99% |
| Treatment data | 71.1% | 78.9% |

### Next Steps
- Session 1.3: Cross-cohort comparison & harmonization roadmap